# Stage 3 -- Dependency-aware deep detectors on S1 (E1, E2)
Trains LGC, LAT, LAT-mask, BR-trunk and the re-implemented WIDENNET on all views under
S1; hyper-parameter search (1 seed) then final runs (3 seeds); family-token ablation on
F_all; saves test probabilities for Stage 5. **Accelerator: GPU (T4/P100).**
**Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`, `scvd-stage2-baselines`.

In [5]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/datasets/smnihalahmed/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage3_deep')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

2026-09-24 19:09:49,071 INFO stage stage3_deep initialised at /kaggle/working/stage3_deep
2026-09-24 19:09:49,071 INFO seeds=(0, 1, 2) quick=False


In [6]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

2026-09-24 19:09:59,883 INFO loaded DIVE: pre=(22330, 222) post=(22330, 177) labels=(22330, 9) tools=(26071, 49)
2026-09-24 19:09:59,885 INFO family code_metadata        6 features
2026-09-24 19:09:59,886 INFO family interface_source    46 features
2026-09-24 19:09:59,887 INFO family creation_opcodes   168 features
2026-09-24 19:09:59,888 INFO family tx_metadata         10 features
2026-09-24 19:09:59,889 INFO family temporal_position    8 features
2026-09-24 19:09:59,889 INFO family runtime_opcodes    158 features
2026-09-24 19:09:59,891 INFO view F_pre  220 features
2026-09-24 19:09:59,891 INFO view F_post 176 features
2026-09-24 19:09:59,892 INFO view F_all  396 features


In [7]:
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)
GRID = [{'lr': lr, 'dm': dm, 'layers': L, 'epochs': (30 if scvd.quick() else 120),
         'batch': 256, 'tau': tau}
        for lr in (1e-3, 3e-4) for dm in (128, 256) for L in (2, 3) for tau in (0.1, 0.3)]
TRIALS = 4 if scvd.quick() else len(GRID)

In [8]:
def build(model_name, vf, M, cfg, n_op):
    if model_name == 'LGC':      return scvd.LGC(vf, M, dm=cfg['dm'], tau=cfg['tau'])
    if model_name == 'LAT':      return scvd.LAT(vf, dm=cfg['dm'], layers=cfg['layers'])
    if model_name == 'LAT-mask': return scvd.LAT(vf, dm=cfg['dm'], layers=cfg['layers'], mask_ll=True)
    if model_name == 'BR-trunk': return scvd.LAT(vf, dm=cfg['dm'], layers=cfg['layers'], trunk_only=True)
    if model_name == 'WIDENNET': return scvd.WIDENNET(n_op, dm=cfg['dm'])

rows, preds = [], {}
for view in scvd.VIEWS:
    d = scvd.make_data(pre, post, lab, fam, view, *splits['S1'])
    M = scvd.cooccurrence(d['y_tr'])
    op_cols = [c for n in scvd.view_families(fam, view) if 'opcodes' in n for c in fam[n]]
    n_op = len(op_cols)
    for name in ['LGC', 'LAT', 'LAT-mask', 'BR-trunk', 'WIDENNET']:
        kind = 'op' if name == 'WIDENNET' else 'fam'
        if kind == 'op':  # flat opcode matrix views
            Xop_tr = np.hstack([d['x_tr'][n] for n in d['x_tr'] if 'opcodes' in n])
            Xop_va = np.hstack([d['x_va'][n] for n in d['x_va'] if 'opcodes' in n])
            Xop_te = np.hstack([d['x_te'][n] for n in d['x_te'] if 'opcodes' in n])
            duse = {'x_tr': Xop_tr, 'x_va': Xop_va, 'x_te': Xop_te,
                    'y_tr': d['y_tr'], 'y_va': d['y_va'], 'y_te': d['y_te']}
        else:
            duse = d
        best_cfg, best_vm = GRID[0], -1
        for cfg in GRID[:TRIALS]:
            mdl = build(name, d['fams'] if kind == 'fam' else None, M, cfg, n_op)
            _, hist = scvd.train_torch(mdl, duse, cfg, log, kind)
            vm = max(h['val_macro_f1'] for h in hist)
            if vm > best_vm: best_vm, best_cfg = vm, cfg
        for seed in SEEDS:
            scvd.set_seed(seed)   # was np.random.seed(seed) -- didn't touch torch's RNG at all
            mdl = build(name, d['fams'] if kind == 'fam' else None, M, best_cfg, n_op)
            pt, hist = scvd.train_torch(mdl, duse, best_cfg, log, kind)
            pv = scvd.predict(mdl, duse, 'va', kind)
            th = scvd.tune_thresholds(d['y_va'], pv)
            m = scvd.all_metrics(d['y_te'], (pt >= th).astype(int), pt)
            rows.append({'model': name, 'view': view, 'split': 'S1', 'seed': seed, **best_cfg, **m})
            scvd.append_results(root, 'stage3_deep', [rows[-1]])   # checkpoint per run
            preds[(name, view, seed)] = (d['ids_te'], pt, th)
            log.info('%-9s F_%-4s seed%d macro_f1=%.4f', name, view, seed, m['macro_f1'])
np.savez_compressed(f'{root}/results/stage3_test_predictions.npz',
                    **{f'{n}__{v}__{s}__ids': i for (n, v, s), (i, _p, _t) in preds.items()},
                    **{f'{n}__{v}__{s}__probs': p for (n, v, s), (_i, p, _t) in preds.items()})
json.dump({f'{n}|{v}|{s}': list(map(float, t)) for (n, v, s), (_i, _p, t) in preds.items()},
          open(f'{root}/results/stage3_thresholds.json', 'w'))

2026-09-24 19:10:21,578 INFO   ep 000 loss 2.5675 val_macro_f1 0.2965 *
2026-09-24 19:10:21,860 INFO   ep 001 loss 1.5223 val_macro_f1 0.3013 *
2026-09-24 19:10:22,134 INFO   ep 002 loss 1.9982 val_macro_f1 0.3683 *
2026-09-24 19:10:22,414 INFO   ep 003 loss 1.1061 val_macro_f1 0.4882 *
2026-09-24 19:10:22,692 INFO   ep 004 loss 0.7295 val_macro_f1 0.5230 *
2026-09-24 19:10:22,963 INFO   ep 005 loss 1.3943 val_macro_f1 0.4462
2026-09-24 19:10:23,244 INFO   ep 006 loss 0.6674 val_macro_f1 0.5463 *
2026-09-24 19:10:23,524 INFO   ep 007 loss 0.6372 val_macro_f1 0.5027
2026-09-24 19:10:23,823 INFO   ep 008 loss 1.1329 val_macro_f1 0.5799 *
2026-09-24 19:10:24,121 INFO   ep 009 loss 0.7759 val_macro_f1 0.6349 *
2026-09-24 19:10:24,411 INFO   ep 010 loss 1.1740 val_macro_f1 0.6247
2026-09-24 19:10:24,705 INFO   ep 011 loss 1.0503 val_macro_f1 0.5621
2026-09-24 19:10:24,995 INFO   ep 012 loss 0.6488 val_macro_f1 0.6353 *
2026-09-24 19:10:25,278 INFO   ep 013 loss 0.4715 val_macro_f1 0.6257
20

In [9]:
drop = {}
d = scvd.make_data(pre, post, lab, fam, 'all', *splits['S1'])
M = scvd.cooccurrence(d['y_tr'])
base = {}
for name in ['LGC', 'LAT']:
    cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'epochs': 60 if scvd.quick() else 120,
           'batch': 256, 'tau': 0.2}
    mdl = build(name, d['fams'], M, cfg, 0)
    pt, _h = scvd.train_torch(mdl, d, cfg, log)
    base[name] = scvd.all_metrics(d['y_te'], (pt >= 0.5).astype(int), pt)['macro_f1']
    for f in d['fams']:
        vf2 = {k: v for k, v in d['fams'].items() if k != f}
        d2 = {k: v for k, v in d.items() if not k.startswith('x_')}
        d2['fams'] = vf2
        for part in ('tr', 'va', 'te'):
            d2['x_' + part] = {k: v for k, v in d['x_' + part].items() if k != f}
        mdl2 = build(name, vf2, M, cfg, 0)
        pt2, _h2 = scvd.train_torch(mdl2, d2, cfg, log)
        drop.setdefault(f, {})[name] = base[name] - scvd.all_metrics(
            d['y_te'], (pt2 >= 0.5).astype(int), pt2)['macro_f1']
json.dump(drop, open(f'{root}/results/stage3_family_ablation.json', 'w'), indent=1)
scvd.plot_family_ablation(drop, root, log)

2026-09-25 00:56:54,864 INFO   ep 000 loss 3.4602 val_macro_f1 0.0920 *
2026-09-25 00:56:55,203 INFO   ep 001 loss 1.4630 val_macro_f1 0.3487 *
2026-09-25 00:56:55,544 INFO   ep 002 loss 2.1574 val_macro_f1 0.1512
2026-09-25 00:56:55,900 INFO   ep 003 loss 1.2102 val_macro_f1 0.3505 *
2026-09-25 00:56:56,253 INFO   ep 004 loss 0.9393 val_macro_f1 0.5655 *
2026-09-25 00:56:56,604 INFO   ep 005 loss 1.0393 val_macro_f1 0.3151
2026-09-25 00:56:56,947 INFO   ep 006 loss 0.9140 val_macro_f1 0.5456
2026-09-25 00:56:57,284 INFO   ep 007 loss 0.7322 val_macro_f1 0.5760 *
2026-09-25 00:56:57,650 INFO   ep 008 loss 0.8809 val_macro_f1 0.5817 *
2026-09-25 00:56:58,006 INFO   ep 009 loss 0.9983 val_macro_f1 0.5051
2026-09-25 00:56:58,378 INFO   ep 010 loss 1.5493 val_macro_f1 0.6302 *
2026-09-25 00:56:58,751 INFO   ep 011 loss 1.0736 val_macro_f1 0.5824
2026-09-25 00:56:59,094 INFO   ep 012 loss 0.7528 val_macro_f1 0.5720
2026-09-25 00:56:59,446 INFO   ep 013 loss 0.5957 val_macro_f1 0.6134
2026-0

In [10]:
summ, sp = scvd.write_summary(root, 'stage3_deep')
scvd.plot_model_macro_f1(summ, root, 's3_deep_macro_f1', log)
scvd.plot_perlabel_f1(summ, ['LGC', 'LAT', 'LAT-mask', 'BR-trunk'], root, 's3_deep_perlabel', log)
scvd.tex_table(summ, f'{root}/tables/stage3_deep.tex',
               ['macro_f1', 'micro_f1', 'macro_ap', 'subset_acc'],
               'Deep detectors on S1 (mean over seeds).', 'tab:stage3deep')
summ.round(4)

2026-09-25 01:13:07,054 INFO maxp pruned
2026-09-25 01:13:07,060 INFO cmap pruned
2026-09-25 01:13:07,061 INFO kern dropped
2026-09-25 01:13:07,062 INFO post pruned
2026-09-25 01:13:07,062 INFO FFTM dropped
2026-09-25 01:13:07,064 INFO GPOS pruned
2026-09-25 01:13:07,066 INFO GSUB pruned
2026-09-25 01:13:07,072 INFO glyf pruned
2026-09-25 01:13:07,073 INFO Added gid0 to subset
2026-09-25 01:13:07,074 INFO Added first four glyphs to subset
2026-09-25 01:13:07,074 INFO Closing glyph list over 'MATH': 35 glyphs before
2026-09-25 01:13:07,075 INFO Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'I', 'L', 'N', 'R', 'T', 'W', 'a', 'c', 'eight', 'four', 'hyphen', 'k', 'm', 'n', 'nonmarkingreturn', 'o', 'one', 'period', 'r', 's', 'six', 'space', 't', 'two', 'u', 'zero']
2026-09-25 01:13:07,076 INFO Glyph IDs:   [0, 1, 2, 3, 16, 17, 19, 20, 21, 23, 25, 27, 36, 37, 38, 39, 40, 41, 42, 44, 47, 49, 53, 55, 58, 68, 70, 78, 80, 81, 82, 85, 86, 87, 88]
2026-09-25 01:13:07,077 INF

,model,view,split,seed,lr,dm,layers,epochs,batch,tau,...,f1_FrontR_sd,f1_TimeMan_sd,auc_Reent_sd,auc_AccCtrl_sd,auc_Arith_sd,auc_UnchkRet_sd,auc_DoS_sd,auc_BadRnd_sd,auc_FrontR_sd,auc_TimeMan_sd
0,BR-trunk,all,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0102,0.0083,0.0004,0.0011,0.0021,0.0024,0.0048,0.0054,0.0137,0.0022
1,BR-trunk,post,S1,1.0,0.001,256.0,3.0,120.0,256.0,0.3,...,0.0430,0.0086,0.0040,0.0202,0.0055,0.0049,0.0080,0.0177,0.0135,0.0027
2,BR-trunk,pre,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0565,0.0077,0.0052,0.0030,0.0026,0.0017,0.0066,0.0015,0.0077,0.0016
3,LAT,all,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0322,0.0090,0.0011,0.0007,0.0040,0.0028,0.0044,0.0133,0.0075,0.0016
4,LAT,post,S1,1.0,0.001,256.0,3.0,120.0,256.0,0.1,...,0.0216,0.0085,0.0060,0.0021,0.0117,0.0038,0.0043,0.0061,0.0108,0.0029
5,LAT,pre,S1,1.0,0.001,256.0,3.0,120.0,256.0,0.1,...,0.0194,0.0017,0.0013,0.0003,0.0015,0.0021,0.0006,0.0104,0.0099,0.0014
6,LAT-mask,all,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0177,0.0049,0.0003,0.0019,0.0023,0.0031,0.0081,0.0056,0.0126,0.0010
7,LAT-mask,post,S1,1.0,0.001,256.0,3.0,120.0,256.0,0.1,...,0.0262,0.0063,0.0039,0.0027,0.0128,0.0064,0.0025,0.0317,0.0166,0.0068
8,LAT-mask,pre,S1,1.0,0.001,128.0,3.0,120.0,256.0,0.3,...,0.0245,0.0021,0.0021,0.0018,0.0016,0.0017,0.0022,0.0028,0.0180,0.0038
9,LGC,all,S1,1.0,0.001,256.0,2.0,120.0,256.0,0.3,...,0.0075,0.0055,0.0015,0.0137,0.0034,0.0041,0.0036,0.0372,0.0094,0.0047


In [11]:
title = 'SCVD Stage 3: deep dependency-aware detectors'  # 46 chars, under the cap
url = scvd.push_dataset(root, 'stage3_deep', 'scvd-stage3-deep', title, log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)

Starting upload for file tables.zip


100%|██████████| 557/557 [00:00<00:00, 893B/s]


Upload successful: tables.zip (557B)
Starting upload for file results.zip


100%|██████████| 4.27M/4.27M [00:00<00:00, 5.99MB/s]


Upload successful: results.zip (4MB)
Starting upload for file plots.zip


100%|██████████| 202k/202k [00:00<00:00, 341kB/s]


Upload successful: plots.zip (202KB)
Starting upload for file logs.zip


100%|██████████| 452k/452k [00:00<00:00, 735kB/s]


Upload successful: logs.zip (452KB)
2026-09-25 01:13:14,279 INFO pushed new version -> https://www.kaggle.com/datasets/smnihalahmed/scvd-stage3-deep
DATASET: https://www.kaggle.com/datasets/smnihalahmed/scvd-stage3-deep
